# Primary-9 bounded-compute MAIN experiment

This notebook evaluates the final test split, only after validation-only checkpoint selection. Run cells top-to-bottom in a fresh model-specific Colab session. No further hyperparameter search. Python 3.12; actual compatible CUDA GPU, not an A100 name requirement. MOIRAI Traffic H720 fine-tuning waits for >=75 GiB. No automatic AMP/channel reduction.

Keep the published execution commit unchanged across restarts. Drive authorization is performed by the user. Completed runs skip; train and test progress resume. Interrupted running.lock needs manual review, never automatic deletion.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from pathlib import Path

assert sys.version_info[:2] == (3, 12), "Select a Python 3.12 Colab runtime"
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
FAMILY = input("Model family (ttm or moirai1): ").strip()
assert FAMILY in ("ttm", "moirai1")
COMMIT = input("Full published main-study execution commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-main-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT
print("Immutable execution commit:", COMMIT)

In [ ]:
from google.colab import drive

# User executes this authorization cell. No credentials are stored in source/output.
drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-main-study")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
# Persist public data only. HF/Xet symlink caches must stay on local disk.
for relative in ("data/source_cache",):
    target = PERSIST / "public-cache" / relative
    target.mkdir(parents=True, exist_ok=True)
    link = ROOT / relative
    link.parent.mkdir(parents=True, exist_ok=True)
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)
# Local model cache guard: never automatically remove an existing Drive cache.
model_cache = ROOT / ".cache"
drive_root = Path("/content/drive").resolve()
for candidate in (model_cache, model_cache / FAMILY, model_cache / "hf-home", model_cache / "xet"):
    resolved = candidate.resolve()
    assert resolved != drive_root and drive_root not in resolved.parents, (
        "Model cache points to Drive; preserve it and review recovery before continuing."
    )
model_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(model_cache / "hf-home")
os.environ["HF_XET_CACHE"] = str(model_cache / "xet")
print("Local model cache (re-download after runtime loss):", model_cache.resolve())
print("Drive results/checkpoints:", OUT)
print("Temporary /content is not persistent. Do not remove previous failure evidence.")

In [ ]:
ENV = Path("/content") / ("venv-main-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "python3.12-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-main.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = """
import json, sys, torch
assert torch.cuda.is_available()
assert sys.version_info[:2] == (3, 12)
print(json.dumps(dict(interpreter=sys.executable, python=sys.version, torch=torch.__version__,
    cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(),
    vram_gib=torch.cuda.get_device_properties(0).total_memory/2**30)))
"""
subprocess.run([str(PY), "-c", probe], check=True)
print("Every probe/training/data subprocess uses", PY, "; kernel imports no vendor package.")

In [ ]:
BASE = [
    str(PY),
    "-m",
    "tsfm_crossover.experiments.main_study",
    "--output",
    str(OUT),
    "--expected-commit",
    COMMIT,
]
subprocess.run(BASE, check=True)
plan = json.loads((OUT / "plan.json").read_text())
assert len(plan["conditions"]) == 1944
DATASET = input("Dataset [ETTh1]: ").strip() or "ETTh1"
HORIZON = int(input("Horizon 96/192/336/720 [96]: ").strip() or "96")
SEED = int(input("Seed 1729/2718/31415 [1729]: ").strip() or "1729")
jobs = [
    r
    for r in plan["conditions"]
    if r["family"] == FAMILY
    and r["dataset"] == DATASET
    and r["horizon"] == HORIZON
    and r["seed"] == SEED
]
assert len(jobs) == 9, "Choose an approved dataset/horizon/seed"
for r in jobs:
    print(r["id"], "completed" if (OUT / r["id"] / "result.json").exists() else "pending")
MAX_CONDITIONS = 9
WALL_HOURS = 6
print("At most nine new/resumed conditions; stops starting jobs after six hours.")
print("Final test scores must NOT be used to modify this fixed protocol.")

In [ ]:
subprocess.run([str(PY), "scripts/prepare_study_data.py", "--dataset", DATASET], check=True)
print("Dataset integrity verified; no test performance used for preparation.")

In [ ]:
assert input("Type RUN_MAIN to execute this authorized final-test shard: ").strip() == "RUN_MAIN"
started = time.monotonic()
executed = 0
for row in jobs:
    if (OUT / row["id"] / "result.json").exists():
        # Still validate identity in the runner before skipping a completed row.
        subprocess.run(BASE + ["--condition-id", row["id"]], check=True)
        continue
    if executed >= MAX_CONDITIONS or time.monotonic() - started >= WALL_HOURS * 3600:
        print("Shard session budget reached; resume later with the same commit and settings.")
        break
    print("Starting", row["id"], flush=True)
    subprocess.run(BASE + ["--condition-id", row["id"]], check=True)
    executed += 1
    result = OUT / row["id"] / "result.json"
    print(
        row["id"],
        json.loads(result.read_text())["status"]
        if result.exists()
        else "RESOURCE PENDING: inspect pending JSON; no success recorded",
        flush=True,
    )
print("Export below even if a condition failed/interrupted. Do not alter hyperparameters.")

In [ ]:
from google.colab import files

stamp = uuid.uuid4().hex[:8]
archive = Path("/content") / f"{FAMILY}-main-{DATASET}-h{HORIZON}-s{SEED}-{stamp}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(OUT / "plan.json", "plan.json")
    for row in jobs:
        directory = OUT / row["id"]
        for p in directory.glob("*.json"):
            if p.name == "training.json":
                continue  # large diagnostic history; persistent on Drive
            assert p.stat().st_size <= 2_000_000, f"Unexpected large JSON: {p.name}"
            z.write(p, p.relative_to(OUT))
saved = PERSIST / "exports" / archive.name
saved.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, saved)
files.download(str(archive))
print("Backup ZIP:", saved)
print("Checkpoint/sampling manifests remain on Drive; never commit them or raw data.")
print("Return this ZIP for CPU provenance validation and cumulative analysis.")

## Continue without changing the protocol

Repeat plan → data → run → export for another predeclared seed/horizon/dataset. Use a **fresh session** for the other model family. An OOM or unavailable GPU is not a performance result. Retain the failure JSON, stop that shard and request compatible hardware; no model/precision/channel fallback. Partial test sums resume on matching hardware and package versions. Hardware changes within a partial condition are blocked and require an explicit reviewed restart in a new run directory. A three-seed/complete-rate group is required for a crossover report.

After abrupt runtime death, inspect the old process/session before manually removing only that condition's stale `running.lock`. Do not delete checkpoints, selection.json or test-progress.json.